# EELSNMF Demo

In [1]:
import EELSNMF as enmf

cupy not available


In [2]:
# The main input for EELSNMF is hyperspy signal and the experimental conditions at which it was acquired:
import hyperspy.api as hs
s = hs.load(r"C:\Users\torruell\Documents\EPFL\eelsnmf tests\Test_SI.hspy")

# EELSNMF if a non-negative matrix factorization framework. The data need to be strictly positive:
s.data = np.clip(s.data,0,None)


# The experimental parameters are read if correctly defined in the image metadata.
deco = enmf.EELSNMF(s)
print(f"beam energy (V): {deco.E0}")
print(f"Convergence angle (rad): {deco.alpha}")
print(f"Collection angle (rad): {deco.beta}")

beam energy (V): 300000.0
Convergence angle (rad): 0.02022
Collection angle (rad): 0.036000000000000004


In [3]:
# For the physics guided approach we need to define the contributions to the measured EELS spectra, which are stored in the G matrix:

deco.build_G(
    low_loss = None, # currently the model only suports convolution with the average low-loss (not pixel-wise for a dual-EELS dataset)

    fine_structure_ranges = { #Here we define what are the edge that we expect in our EELS spectrum and their ELNES ranges (in eV)
        "Fe_L" : (704.,732.),
        "Cr_L" : (571.,597.),
        "O_K" : (526.,560.)
    }
)

deco.plot_energy_ranges()

L2 is used
L1 is used
L2 is used
L1 is used


At this stage we are ready for spectral decomposition.
There are different methods to choose depending on if smoothness and/or Sum-rule conservation want to be enforced.
The default method solves:

$$
\arg\min_{W,H}  \| X - G W H \|_F^2
$$

Using NMF multiplicative updates.

In [4]:
deco.decomposition(
    n_components = 2,
    max_iters = 200,
    tol = 1e-9, # This tolerance defines the convergence condition
    decomposition_method = "default_decomposition"
    
)

100%|███████████████████████████████████████████| 200/200 [00:05<00:00, 33.70it/s, error=3.47e+6, relative change=4e-5]


In [ ]:
#there are several built in plots to asses the results:
deco.plot_average_model() # plots the match between the average data spectrum and average model

In [8]:
deco.plot_factors() # plots the spectrum of each phase (the columns in GW)

In [7]:
deco.plot_loadings() # plots the abundance of each phase (the rows in H)

In [ ]:
deco.plot_edges() # plots the ELNES associated to each element in each phase

In [6]:
deco.plot_chemical_maps()